# Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load Merged Data

In [2]:
df = pd.read_csv(r"..\..\Data\Main\merged_data.csv")
df['date'] = pd.to_datetime(df['date'])
print(f"Shape: {df.shape}")
print(f"Tickers: {df['ticker'].nunique()}")
print(df.head())

Shape: (589886, 8)
Tickers: 523
        date  sentiment ticker      Close       High        Low       Open  \
0 2020-01-02     0.0000      A  82.210472  82.593070  81.493103  82.162652   
1 2020-01-03     0.0000      A  80.890488  81.617423  80.823534  80.986135   
2 2020-01-07     0.3292      A  81.378319  81.550488  80.287919  80.307045   
3 2020-01-08     0.0000      A  82.181747  82.707821  81.493072  82.220008   
4 2020-01-09    -0.6269      A  83.473022  83.884314  82.420883  82.698267   

    Volume  
0  1410500  
1  1118300  
2  1684700  
3  1847600  
4  1912700  


# Building Required Features

## Log Returns

In [3]:
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
df['log_return'] = df.groupby('ticker')['Close'].transform(lambda x: np.log(x / x.shift(1)))

print(f"log_return NaN: {df['log_return'].isna().sum()} (expected: {df['ticker'].nunique()} — one per ticker)")
print(f"Mean: {df['log_return'].mean():.6f}")
print(f"Std: {df['log_return'].std():.6f}")

log_return NaN: 523 (expected: 523 — one per ticker)
Mean: 0.000360
Std: 0.025145


## 20-Day Rolling Volatility

In [4]:
df['volatility_20d'] = df.groupby('ticker')['log_return'].transform(lambda x: x.rolling(20).std())

print(f"volatility_20d NaN: {df['volatility_20d'].isna().sum()}")
print(f"Mean: {df['volatility_20d'].mean():.6f}")
print(f"Std: {df['volatility_20d'].std():.6f}")

volatility_20d NaN: 10460
Mean: 0.021540
Std: 0.013159


## Next-Day Return and Binary Target

In [5]:
df['next_day_return'] = df.groupby('ticker')['log_return'].shift(-1)
df['target'] = (df['next_day_return'] > 0).astype(int)

print(f"next_day_return NaN: {df['next_day_return'].isna().sum()} (expected: {df['ticker'].nunique()} — last day per ticker)")
print(f"\nTarget class balance:")
print(df['target'].value_counts(normalize=True).round(4))

next_day_return NaN: 523 (expected: 523 — last day per ticker)

Target class balance:
target
1    0.5173
0    0.4827
Name: proportion, dtype: float64


## Interaction Term and Market Volatility Features

In [6]:
# Sentiment × Volatility interaction term
df['sent_x_vol'] = df['sentiment'] * df['volatility_20d']

# Market-wide volatility (daily average across all stocks)
df['market_vol'] = df.groupby('date')['volatility_20d'].transform('mean')

# Binary volatile market indicator (above median = volatile)
overall_median = df['market_vol'].median()
df['volatile_market'] = (df['market_vol'] > overall_median).astype(int)

print(f"sent_x_vol NaN: {df['sent_x_vol'].isna().sum()}")
print(f"market_vol NaN: {df['market_vol'].isna().sum()}")
print(f"Market volatility median: {overall_median:.6f}")
print(f"\nVolatile market split:")
print(df['volatile_market'].value_counts(normalize=True).round(4))

sent_x_vol NaN: 10460
market_vol NaN: 6433
Market volatility median: 0.019225

Volatile market split:
volatile_market
0    0.5061
1    0.4939
Name: proportion, dtype: float64


# Save Modelling Dataset

In [7]:
print(f"=== Final Modelling Dataset ===")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Tickers: {df['ticker'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

df.to_csv(r"..\..\Data\Main\modelling_dataset.csv", index=False)
print(f"\nSaved: modelling_dataset.csv")

=== Final Modelling Dataset ===
Shape: (589886, 15)
Columns: ['date', 'sentiment', 'ticker', 'Close', 'High', 'Low', 'Open', 'Volume', 'log_return', 'volatility_20d', 'next_day_return', 'target', 'sent_x_vol', 'market_vol', 'volatile_market']
Tickers: 523
Date range: 2020-01-02 to 2025-12-31

Saved: modelling_dataset.csv
